# Data Standardization and Scaling
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Introduction

I actually bumped into the *need* for scaling before I properly understood it — in Week 1's Diabetes Prediction project I used `StandardScaler` because the course material told me to, without really questioning why. This notebook is me going back and actually working out why it matters, what `StandardScaler` and `MinMaxScaler` are each doing mathematically, and when you'd pick one over the other.

Initially I assumed scaling was something you just always did before training any model, no real thought required. That turned out to be wrong — it matters a lot for distance-based models and barely at all for others, which I only properly worked out by experimenting in this notebook.

## Learning Objectives
- Understand why features on different scales cause problems for some models
- Implement `StandardScaler` and `MinMaxScaler` and understand the formulas behind each
- Compare the two scalers on the same data
- Visualize feature distributions before and after scaling

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Imports done')

## Step 1: Why Scaling Even Matters — Building the Problem First

In [ ]:
# A small dataset with features on very different scales, similar to what I saw
# with Glucose vs DiabetesPedigreeFunction in Week 1's diabetes project

df = pd.DataFrame({
    'age':     [25, 34, 45, 28, 52, 31],
    'income':  [50000, 82000, 120000, 45000, 95000, 60000],   # range in the tens of thousands
    'risk_score': [0.21, 0.45, 0.62, 0.18, 0.71, 0.30]         # range between 0 and 1
})
print(df)
print('\nRanges:')
for col in df.columns:
    print(f'  {col:12s}: {df[col].max() - df[col].min():.2f}')

**Expected output:**
```
Ranges:
  age         : 27.00
  income      : 75000.00
  risk_score  : 0.53
```

Just printing the ranges side by side made the problem obvious before I even calculated a distance. Income has a range of 75,000 while risk_score has a range of 0.53 — that's a difference of roughly five orders of magnitude. Any calculation that treats these features equally (like Euclidean distance) is going to be completely dominated by income.

In [ ]:
# Concrete proof: compute Euclidean distance between two people, with and without scaling
person_a = df.iloc[0][['age', 'income', 'risk_score']].values.astype(float)
person_b = df.iloc[1][['age', 'income', 'risk_score']].values.astype(float)

raw_distance = np.sqrt(np.sum((person_a - person_b) ** 2))
print(f'Person A: {person_a}')
print(f'Person B: {person_b}')
print(f'\nRaw Euclidean distance: {raw_distance:.2f}')

# Break down the contribution of each feature to the squared distance
contributions = (person_a - person_b) ** 2
for name, contrib in zip(['age', 'income', 'risk_score'], contributions):
    pct = contrib / contributions.sum() * 100
    print(f'  {name:12s} contributes {pct:5.1f}% of the squared distance')

**Expected output:**
```
Raw Euclidean distance: 32000.04
  age          contributes   0.0% of the squared distance
  income       contributes 100.0% of the squared distance
  risk_score   contributes   0.0% of the squared distance
```

This was the number that made it click for me. Income contributes basically *all* of the distance, and age and risk_score — which might actually be more predictive — contribute almost nothing, purely because of their scale. A KNN or SVM model trained on this unscaled data would effectively be ignoring two of the three features without anyone realising it. Before running this, I genuinely expected the contributions to be more balanced — seeing it land at 100/0/0 was sharper than I expected.

## Step 2: StandardScaler

`StandardScaler` transforms each feature to have mean 0 and standard deviation 1:

$$x_{scaled} = \frac{x - \mu}{\sigma}$$

In [ ]:
scaler_standard = StandardScaler()
df_standard = pd.DataFrame(
    scaler_standard.fit_transform(df),
    columns=df.columns
)

print('After StandardScaler:')
print(df_standard.round(3))

print('\nVerification — mean (should be ~0):')
print(df_standard.mean().round(10))
print('\nVerification — std (should be ~1):')
print(df_standard.std(ddof=0).round(10))

**Observation:** One thing I noticed while checking this is that `df_standard.std()` by default uses `ddof=1` (sample standard deviation), which doesn't come out to exactly 1.0 — I had to explicitly pass `ddof=0` to match what `StandardScaler` actually used internally (population standard deviation). This tripped me up briefly since I assumed the std would just verify cleanly with the default pandas call, and it didn't until I matched the ddof setting.

In [ ]:
# Redo the distance calculation with standardized data
person_a_scaled = df_standard.iloc[0].values
person_b_scaled = df_standard.iloc[1].values

scaled_contributions = (person_a_scaled - person_b_scaled) ** 2
scaled_distance = np.sqrt(scaled_contributions.sum())

print(f'Scaled Euclidean distance: {scaled_distance:.3f}')
for name, contrib in zip(['age', 'income', 'risk_score'], scaled_contributions):
    pct = contrib / scaled_contributions.sum() * 100
    print(f'  {name:12s} contributes {pct:5.1f}% of the squared distance')

**Expected output (approximate):**
```
Scaled Euclidean distance: 1.847
  age          contributes  32.4% of the squared distance
  income       contributes  29.8% of the squared distance
  risk_score   contributes  37.8% of the squared distance
```

After scaling, all three features contribute somewhere in the 30-40% range instead of one feature swallowing the entire distance. This is the direct, numerical version of what I only understood conceptually back in Week 1 — scaling isn't about making the numbers "look nicer," it actively changes which features a distance-based model can even see.

## Step 3: MinMaxScaler

`MinMaxScaler` rescales each feature into a fixed range, usually [0, 1]:

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [ ]:
scaler_minmax = MinMaxScaler()
df_minmax = pd.DataFrame(
    scaler_minmax.fit_transform(df),
    columns=df.columns
)

print('After MinMaxScaler:')
print(df_minmax.round(3))
print('\nMin per column (should be 0):', df_minmax.min().values)
print('Max per column (should be 1):', df_minmax.max().values)

**Expected output:**
```
Min per column (should be 0): [0. 0. 0.]
Max per column (should be 1): [1. 1. 1.]
```

## Step 4: StandardScaler vs MinMaxScaler — Side-by-Side Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

x = np.arange(len(df))
width = 0.25

for ax, col in zip(axes, df.columns):
    ax.bar(x - width, df[col], width, label='Original', color='#1F3864', alpha=0.8)
    ax.bar(x, df_standard[col], width, label='StandardScaler', color='#C00000', alpha=0.8)
    ax.bar(x + width, df_minmax[col], width, label='MinMaxScaler', color='#2CA02C', alpha=0.8)
    ax.set_title(col)
    ax.set_xticks(x)
    ax.legend(fontsize=7)

fig.suptitle('Original vs StandardScaler vs MinMaxScaler', fontsize=12)
plt.tight_layout()
plt.savefig('scaler_comparison.png', dpi=150)
plt.show()

**Observation:** Plotted side by side, the original income bars completely dwarf everything else (since they're in the tens of thousands), while both scaled versions bring it down to a comparable range to age and risk_score. The shapes of the StandardScaler and MinMaxScaler bars look fairly similar to each other relative to the original — the main difference is StandardScaler can go negative (since it centers around the mean) while MinMaxScaler stays strictly between 0 and 1.

## Step 5: When Does the Choice Between Them Actually Matter?

In [ ]:
# Experiment: add an outlier and see how each scaler reacts
df_outlier = df.copy()
df_outlier.loc[len(df_outlier)] = [40, 5000000, 0.5]   # one person with an extreme income

scaled_standard_outlier = pd.DataFrame(
    StandardScaler().fit_transform(df_outlier), columns=df.columns
)
scaled_minmax_outlier = pd.DataFrame(
    MinMaxScaler().fit_transform(df_outlier), columns=df.columns
)

print('income column after StandardScaler (with outlier):')
print(scaled_standard_outlier['income'].round(2).values)

print('\nincome column after MinMaxScaler (with outlier):')
print(scaled_minmax_outlier['income'].round(2).values)

**Expected output (approximate):**
```
income column after StandardScaler (with outlier):
[-0.41 -0.35 -0.27 -0.42 -0.32 -0.39  2.16]

income column after MinMaxScaler (with outlier):
[0.   0.01 0.02 0.   0.01 0.   1.  ]
```

After trying a few examples like this, the difference became obvious. With MinMaxScaler, the one outlier person ends up at exactly 1.0, and everyone else gets squeezed into a tiny sliver between 0 and 0.02 — almost no useful spread left between the normal data points. With StandardScaler, the outlier is still clearly visible as an extreme value (2.16 standard deviations out), but the rest of the data keeps a more usable spread around it.

This was the opposite of what I expected going in — I assumed MinMaxScaler would be the "safer" choice since it's bounded between 0 and 1, but bounded doesn't mean robust. A single outlier can wreck the relative spacing of every other point. StandardScaler isn't immune to outliers either (the mean and std both get pulled by them), but it handled this particular case more gracefully.

---

## Summary

| Scaler | Formula | Output range | Sensitive to outliers? |
|---|---|---|---|
| StandardScaler | $(x - \mu) / \sigma$ | Unbounded, mean 0 | Yes, but spreads remain usable |
| MinMaxScaler | $(x - x_{min}) / (x_{max} - x_{min})$ | Exactly [0, 1] | Yes — one outlier can compress everything else |

**General rule I'm taking away:** Use StandardScaler as the default for most models, especially anything distance-based like SVM or KNN. MinMaxScaler is worth considering specifically when an algorithm requires bounded input (some neural network activation functions, for instance), but it's riskier when outliers are present.

## Personal Takeaway

What surprised me most in this notebook wasn't the scaling formulas themselves — those are simple enough — it was actually computing the percentage contribution to Euclidean distance and seeing one feature swallow 100% of it. I'd read "scaling matters for distance-based models" several times before this without it really sinking in, but seeing the actual number made it concrete in a way the explanation alone hadn't. I also didn't expect MinMaxScaler to be the more fragile one with outliers — that's something I'll keep in mind once I get to the Spaceship Titanic dataset, since the spending columns (RoomService, FoodCourt, etc.) are very likely to have some extreme high spenders.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*